# Conv1D autoencoder baseline

This notebook is a thin experiment dashboard. Reusable data preparation, modeling, training, metrics, checkpointing, evaluation, and latent export live in `Code/IAFlow` and the command-line scripts. The primary model compresses each `log10(A_theta)` surface from `(31, 101)` to exactly two latent variables without skip connections.

Model selection uses validation data only. Keep the test split untouched until the architecture and hyperparameters are frozen.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Code').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'Code'))

from IAFlow.Artifacts import load_autoencoder_checkpoint
from IAFlow.AutoEncoder import Conv1dAutoEncoder
from IAFlow.Config import load_experiment_config
from IAFlow.Data import CachedSurfaceDataset, load_cache_metadata

CONFIG_PATH = PROJECT_ROOT / 'Config' / 'AutoEncoderConv1D.yml'
config = load_experiment_config(CONFIG_PATH, project_root=PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('Project:', PROJECT_ROOT)
print('Configured latent dimension:', config.model.latent_dim)

## 1. Prepare the contiguous training cache once

The authoritative HDF5 file is optimized for generation and compressed storage, not shuffled row access. The preparation command reads it once and writes memory-mappable `float32` log surfaces for each fixed split. It also computes the mean surface and global RMS from training rows only.

In [ ]:
cache_directory = config.resolve_path(config.data.cache_directory)
metadata_path = cache_directory / 'Metadata.json'
if metadata_path.exists():
    metadata = load_cache_metadata(cache_directory)
    print(json.dumps(metadata, indent=2))
else:
    print('Cache not prepared. Run:')
    print(f'python {PROJECT_ROOT / "Code" / "PrepareData.py"} --config {CONFIG_PATH}')

## 2. Inspect the architecture

The 31 redshift samples are channels and convolution runs along the 101-point wavenumber axis. Every reconstruction must pass through the two-number bottleneck.

In [ ]:
model = Conv1dAutoEncoder(config.model, config.data.input_shape)
print(json.dumps(model.architecture_summary(), indent=2))
try:
    from torchinfo import summary
    summary(model, input_size=(2, *config.data.input_shape), device='cpu')
except ImportError:
    print(model)

## 3. Train from the command line

Long training jobs are intentionally launched through the script so they survive notebook UI interruptions and produce complete, reproducible artifacts. Adjust architecture and hyperparameters in `Config/AutoEncoderConv1D.yml`.

In [ ]:
train_command = (
    f'python {PROJECT_ROOT / "Code" / "TrainAutoEncoder.py"} '
    f'--config {CONFIG_PATH}'
)
print(train_command)
print('Quick smoke option: add --epochs 2 --maximum-train-samples 1024')


## 4. Review a completed run

Set `RUN_DIRECTORY` to a completed experiment. The plots use training and validation history only.

In [ ]:
latest_pointer = config.resolve_path(config.output.root_directory) / 'LatestRun.txt'
RUN_DIRECTORY = Path(latest_pointer.read_text().strip()) if latest_pointer.exists() else None
if RUN_DIRECTORY is None:
    print('No completed run is registered yet.')
else:
    history = json.loads((RUN_DIRECTORY / 'History.json').read_text())
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    validation_loss = [row['validation']['normalized_mse'] for row in history]
    validation_variance = [row['validation']['variance_recovered'] for row in history]

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].semilogy(epochs, train_loss, label='training objective')
    axes[0].semilogy(epochs, validation_loss, label='validation MSE')
    axes[0].set(xlabel='epoch', ylabel='normalized loss')
    axes[0].legend()
    axes[1].plot(epochs, validation_variance)
    axes[1].axhline(0.999, color='black', linestyle='--', label='99.9% target')
    axes[1].set(xlabel='epoch', ylabel='validation variance recovered')
    axes[1].legend()
    figure.tight_layout()

## 5. Inspect held-out reconstructions

This cell uses validation examples. It displays log-space truth, reconstruction, and residual on the physical `(z, k)` grid.

In [ ]:
if RUN_DIRECTORY is not None:
    trained_model, normalization, checkpoint = load_autoencoder_checkpoint(RUN_DIRECTORY / 'Best.pt')
    validation_data = CachedSurfaceDataset(cache_directory, 'validation')
    target_normalized = validation_data[0].unsqueeze(0)
    with torch.inference_mode():
        reconstructed_normalized = trained_model(target_normalized)
    target = normalization.denormalize(target_normalized[0].numpy())
    reconstructed = normalization.denormalize(reconstructed_normalized[0].numpy())
    residual = reconstructed - target

    figure, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    common = dict(aspect='auto', origin='lower')
    image0 = axes[0].imshow(target, **common)
    image1 = axes[1].imshow(reconstructed, **common)
    limit = np.max(np.abs(residual))
    image2 = axes[2].imshow(residual, vmin=-limit, vmax=limit, cmap='coolwarm', **common)
    for axis, title in zip(axes, ['truth log10(A_theta)', 'reconstruction', 'residual']):
        axis.set(title=title, xlabel='wavenumber index', ylabel='redshift channel')
    figure.colorbar(image0, ax=axes[0])
    figure.colorbar(image1, ax=axes[1])
    figure.colorbar(image2, ax=axes[2])

## 6. Freeze, test once, and export latents

After choosing the final architecture from validation results, evaluate `Best.pt` once on the test split, then export ordered train/validation/test latents for the normalizing-flow stage.

```bash
python Code/EvaluateAutoEncoder.py --checkpoint Runs/AutoEncoder/<run>/Best.pt --split test
python Code/ExportLatents.py --checkpoint Runs/AutoEncoder/<run>/Best.pt
```